1. Load raw dataset

In [0]:
df_dengue_spatial_raw = spark.table("raw_dengue_espacial")

display(df_dengue_spatial_raw)
print(df_dengue_spatial_raw.columns)

2. Clean and standardize

In [0]:
from pyspark.sql.functions import col, lit, trim, upper
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Criar índice de linha para remover cabeçalho lixo
window_spec = Window.orderBy(lit(1))

df_dengue_spatial_clean = (
    df_dengue_spatial_raw
    .withColumn("row_id", row_number().over(window_spec))
    .filter(col("row_id") > 3)  # remove as 3 primeiras linhas
    .drop("row_id")
)

# Renomear colunas corretamente
df_dengue_spatial = (
    df_dengue_spatial_clean
    .select(
        col("_c1").alias("data"),
        col("_c2").cast("int").alias("idade"),
        col("_c3").alias("sexo"),
        col("_c4").alias("bairro"),
        col("_c6").alias("exame")
    )
    .withColumn("doenca", lit("DENGUE"))
    .withColumn("bairro", trim(upper(col("bairro"))))
)

3. Save

In [0]:
df_dengue_spatial.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fato_dengue_espacial")

4. Validate

In [0]:
df_fato_dengue_spatial = spark.table("fato_dengue_espacial")

display(df_fato_dengue_spatial)
print(df_fato_dengue_spatial.count())
print(df_fato_dengue_spatial.columns)